In [2]:
import pandas as pd
# 假设你已经生成了一些数据，存储在一个列表中
data = [1, 2, 3, 4, 5]
# 创建一个空的DataFrame用于存储数据
df = pd.DataFrame()
# 使用for循环遍历数据列表，并将每个数据添加到DataFrame中的新行
for item in data:
    # 创建一个新的DataFrame行，使用字典格式传递数据
    row = pd.DataFrame({'Data': [item]})
    # 将新行添加到DataFrame中
    df = pd.concat([df, row], ignore_index=True)
# 将DataFrame保存为Excel文件
df.to_excel('data.xlsx', index=False)

In [5]:
import pandas as pd
import numpy as np
from deap import algorithms, base, creator, tools
df_particles_list = []
for i in range(40):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    from sklearn.utils import shuffle
    from sklearn import svm
    import pyswarms as ps
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import mean_absolute_error, r2_score
    from sklearn.svm import SVR
    from sklearn.model_selection import LeaveOneOut
    from sklearn.metrics import mean_squared_error
    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import r2_score
    from deap import algorithms, base, creator, tools
    # Read data
    data = pd.read_excel('聚类分析2.xlsx')  # Please replace with your data file name
    data = data.drop('合金的牌号', axis=1)
    data = data.round(2)
    elements = ['Ni', 'Cr', 'Co', 'Fe', 'Al', 'Ti', 'Nb', 'Mo', 'W', 'C', 'B', 'Zr', '温度', '应力', '固溶处理温度',
                '固溶处理时间', '强化相溶解温度', '稳定时效温度', '稳定时效时间', '时效温度', '时效时间']

    X = np.zeros((data.shape[0], len(elements)))
    for i, row in data.iterrows():
        for j, el in enumerate(elements):
            X[i, j] = row[el]
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)
    Y = data[['蠕变时间']].values
    Y = np.log(Y)
    def evaluate(true_labels, pred_labels):
        errors = abs(pred_labels - true_labels)
        MAE = mean_absolute_error(true_labels, pred_labels)
        mape = 100 * np.mean(errors / true_labels)
        r2 = r2_score(true_labels, pred_labels)
        RMSE = mean_squared_error(true_labels, pred_labels, squared=False)
        return mape, MAE, RMSE, r2
    model_0 = SVR(C=37.722551111111111111, kernel='rbf', gamma=0.111111111111111111)
    cv = LeaveOneOut()
    Model = model_0
    X = X.reshape(-1, len(elements))
    Model.fit(X, Y)
    from sklearn.model_selection import cross_val_predict
    y_pred = cross_val_predict(Model, X, Y, cv=cv)
    model_accuracy1, model_MAE1, model_RMSE1, model_r21 = evaluate(Y, y_pred)
    print("R^2 score:", model_r21)
    def fitness_func(individual, X, Y, model):
        X_individual = np.tile(individual, (X.shape[0], 1))
        Y_pred = model.predict(X_individual)
        mse = mean_squared_error(Y, Y_pred)
        return (1 / (1 + mse),)
    # Define GA parameters
    POP_SIZE = 400
    NUM_GEN = 40
    CXPB = 0.9
    MUTPB = 0.01
    # Create a fitness function for maximizing fitness values
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)
    # Initialize toolbox
    toolbox = base.Toolbox()
    # Define attributes
    toolbox.register("attr_float", np.random.uniform, low=0, high=1)
    # Define individual and population
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=len(elements))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    # Define evaluation function
    toolbox.register("evaluate", fitness_func, X=X, Y=Y, model=Model)
    # Define genetic operators
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
    toolbox.register("select", tools.selTournament, tournsize=3)
    # Perform genetic algorithm
    population = toolbox.population(n=POP_SIZE)
    hall_of_fame = tools.HallOfFame(maxsize=1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    population, logbook = algorithms.eaSimple(population, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NUM_GEN,
                                              stats=stats, halloffame=hall_of_fame, verbose=True)
    # Extract best individual and its fitness
    best_individual = hall_of_fame[0]
    best_fitness = best_individual.fitness.values[0]
    # Reshape best_individual
    best_particles = np.asarray(best_individual)
    print("------------------------------------------------------------------------------------")
    print("Best particles:", best_particles)
    print("Best fitness:", best_fitness)
    print("------------------------------------------------------------------------------------")
    df_particles = pd.DataFrame([best_particles], columns=elements)
    df_particles_list.append(df_particles)
df_particles_combined = pd.concat(df_particles_list, ignore_index=True)
df_particles_combined.to_excel('best_particles.xlsx', index=False)

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

R^2 score: 0.8804390625989749
gen	nevals	avg     	min     
0  	400   	0.332257	0.133074
1  	358   	0.374831	0.165616
2  	362   	0.381342	0.190973
3  	376   	0.385161	0.177849
4  	359   	0.388588	0.185991
5  	347   	0.388456	0.193987
6  	370   	0.394691	0.211073
7  	352   	0.398182	0.206238
8  	354   	0.398006	0.176505
9  	355   	0.400684	0.151696
10 	377   	0.400835	0.255544
11 	366   	0.403535	0.271414
12 	340   	0.404411	0.250286
13 	358   	0.406833	0.276675
14 	374   	0.403594	0.225259
15 	363   	0.403962	0.240113
16 	358   	0.403708	0.2611  
17 	350   	0.405225	0.285662
18 	377   	0.405369	0.241569
19 	349   	0.408477	0.297807
20 	362   	0.408545	0.313305
21 	362   	0.406481	0.248821
22 	371   	0.406823	0.281768
23 	346   	0.408639	0.27939 
24 	368   	0.410385	0.287665
25 	355   	0.410884	0.280576
26 	341   	0.409855	0.264769
27 	351   	0.411621	0.241054
28 	366   	0.410927	0.232974
29 	364   	0.412273	0.352401
30 	368   	0.413336	0.370667
31 	338   	0.412191	0.284522
32 	372   	0.

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

R^2 score: 0.8804390625989749
gen	nevals	avg     	min      
0  	400   	0.336273	0.0975709
1  	364   	0.373341	0.159714 
2  	364   	0.384212	0.206877 
3  	347   	0.389418	0.208897 
4  	348   	0.389788	0.178705 
5  	349   	0.393192	0.203565 
6  	354   	0.393599	0.180815 
7  	358   	0.394101	0.221369 
8  	360   	0.395492	0.150363 
9  	366   	0.397494	0.191981 
10 	364   	0.3993  	0.243015 
11 	360   	0.399819	0.165748 
12 	368   	0.402472	0.242564 
13 	368   	0.402248	0.251961 
14 	358   	0.404487	0.29875  
15 	356   	0.404534	0.253916 
16 	356   	0.406518	0.279431 
17 	351   	0.405516	0.285235 
18 	379   	0.405204	0.274059 
19 	352   	0.405919	0.300928 
20 	353   	0.407479	0.283851 
21 	364   	0.407684	0.326271 
22 	364   	0.406648	0.297603 
23 	366   	0.407666	0.279193 
24 	350   	0.407912	0.300339 
25 	364   	0.408584	0.301397 
26 	360   	0.408267	0.32318  
27 	367   	0.408153	0.296237 
28 	362   	0.409216	0.25854  
29 	369   	0.41085 	0.347165 
30 	358   	0.409039	0.327149 
31 	365   

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

...

gen	nevals	avg     	min     
0  	400   	0.333481	0.119286
1  	361   	0.369708	0.139467
2  	353   	0.379784	0.175639
3  	377   	0.38107 	0.19022 
4  	360   	0.385945	0.200493
5  	376   	0.385157	0.161836
6  	359   	0.393576	0.169718
7  	360   	0.391772	0.204805
8  	362   	0.390406	0.215521
9  	368   	0.391746	0.219988
10 	342   	0.397037	0.224291
11 	356   	0.398251	0.261227
12 	361   	0.396327	0.238309
13 	370   	0.396626	0.260987
14 	354   	0.399456	0.291641
15 	364   	0.4003  	0.27992 
16 	350   	0.400494	0.219112
17 	357   	0.401978	0.266359
18 	375   	0.404114	0.29971 
19 	345   	0.404563	0.278286
20 	362   	0.40301 	0.244046
21 	358   	0.405596	0.269515
22 	343   	0.404791	0.307674
23 	366   	0.407485	0.305476
24 	357   	0.405207	0.296229
25 	346   	0.404169	0.297445
26 	378   	0.406773	0.324939
27 	358   	0.404808	0.29699 
28 	360   	0.407633	0.343705
29 	363   	0.407245	0.2897  
30 	367   	0.406427	0.289746
31 	362   	0.408603	0.273826
32 	348   	0.410459	0.251162
33 	362   	0.4

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

gen	nevals	avg    	min     
0  	400   	0.33532	0.120576
1  	360   	0.375265	0.182029
2  	344   	0.385367	0.168303
3  	370   	0.387511	0.178084
4  	353   	0.390961	0.16224 
5  	360   	0.389985	0.175503
6  	363   	0.394539	0.263052
7  	348   	0.390855	0.204607
8  	361   	0.389039	0.218404
9  	344   	0.396842	0.213395
10 	354   	0.394795	0.176026
11 	360   	0.399445	0.236158
12 	364   	0.398109	0.22225 
13 	372   	0.398127	0.254107
14 	366   	0.398445	0.275464
15 	360   	0.398732	0.29128 
16 	366   	0.400032	0.213255
17 	370   	0.398665	0.248372
18 	372   	0.397239	0.240075
19 	353   	0.397483	0.268002
20 	352   	0.401688	0.266519
21 	359   	0.402469	0.281835
22 	366   	0.402588	0.277187
23 	366   	0.404026	0.249242
24 	358   	0.405863	0.242886
25 	361   	0.406392	0.277337
26 	353   	0.408537	0.273029
27 	372   	0.4096  	0.31522 
28 	362   	0.408748	0.310617
29 	360   	0.410749	0.354966
30 	364   	0.411356	0.263716
31 	358   	0.411504	0.291016
32 	342   	0.412284	0.265893
33 	370   	0.411

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:1143: DataConversionWarning: A column-vector y was passed when 

gen	nevals	avg     	min     
0  	400   	0.336371	0.119379
1  	368   	0.378972	0.172337
2  	374   	0.388228	0.175588
3  	361   	0.389136	0.204205
4  	374   	0.390969	0.210411
5  	369   	0.394116	0.198354
6  	368   	0.39589 	0.256722
7  	363   	0.396442	0.178448
8  	366   	0.391436	0.19511 
9  	360   	0.398896	0.197433
10 	358   	0.397879	0.18259 
11 	372   	0.40024 	0.22471 
12 	359   	0.398684	0.222881
13 	358   	0.400488	0.212063
14 	360   	0.398085	0.216194
15 	352   	0.397597	0.1645  
16 	348   	0.403467	0.246366
17 	347   	0.402267	0.242368
18 	358   	0.403457	0.227503
19 	348   	0.401797	0.211155
20 	348   	0.40439 	0.238309
21 	362   	0.406185	0.246236
22 	350   	0.406569	0.264988
23 	378   	0.404423	0.242768
24 	358   	0.408009	0.292703
25 	367   	0.405266	0.265694
26 	357   	0.407588	0.245606
27 	360   	0.410407	0.246162
28 	372   	0.409583	0.335406
29 	368   	0.409153	0.272201
30 	367   	0.410145	0.258882
31 	358   	0.411876	0.338504
32 	354   	0.411968	0.353326
33 	349   	0.4